# V2a-RSN 220119_F2_run11: Load, Preprocess, and Window

This dataset-level notebook creates the immutable recording and temporal-window checkpoints shared by every method lane.
Confirm the raw-data path, array keys, sampling rate, behavior mapping, valid ranges, and gaps in the named dataset config before running.


In [ ]:
from omegaconf import OmegaConf

from effectome.data_module import PreprocessConfig, WindowConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_stage_status,
    run_preprocessing,
    stage_artifact_path,
)

DATASET_ID = "220119_F2_run11"
DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220119_F2_run11"
RUN_ID = "reference"
METHOD = "shared"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
DATA_CONFIG_PATH = PROJECT_ROOT / "conf" / "data" / "220119_F2_run11.yaml"
FORCE = False

DATA_CFG = OmegaConf.to_container(OmegaConf.load(DATA_CONFIG_PATH), resolve=True)
if not isinstance(DATA_CFG, dict):
    raise TypeError(f"Expected a mapping in {DATA_CONFIG_PATH}")
DATA_CFG["path"] = str((PROJECT_ROOT / str(DATA_CFG["path"])).resolve())
if str(DATA_CFG.get("dataset_id", "")) != DATASET_OUTPUT_ID:
    raise ValueError(
        f"Notebook output dataset id {DATASET_OUTPUT_ID} does not match config dataset id "
        f"{DATA_CFG.get('dataset_id')}"
    )
if str(DATA_CFG.get("recording_id", "")) != RECORDING_ID:
    raise ValueError(
        f"Notebook output recording id {RECORDING_ID} does not match config recording id "
        f"{DATA_CFG.get('recording_id')}"
    )

PREPROCESS_CFG = PreprocessConfig(
    detrend=False,
    zscore=False,
    deconvolve=False,
    smooth_window=0,
    drop_low_variance=0.0,
)

WINDOW_CFG = WindowConfig(
    length=500,
    history_length=500,
    target_length=15,
    stride=15,
    mode="temporal",
    behavior_summary="mean",
    respect_boundaries=True,
    drop_incomplete_tail=True,
    standardize_per_window=True,
    standardization_epsilon=1.0e-8,
)
EXPECTED_BEHAVIOR_KEYS = ["continuous"]

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_stage_status(run)


In [ ]:
recording, windows = run_preprocessing(
    run,
    data_cfg=DATA_CFG,
    preprocess_cfg=PREPROCESS_CFG,
    window_cfg=WINDOW_CFG,
    force=FORCE,
)
missing_behavior = sorted(set(EXPECTED_BEHAVIOR_KEYS) - set(recording.behavior))
if missing_behavior:
    raise KeyError(f"Dataset is missing configured behavior keys: {missing_behavior}")

print(f"recording: {recording.n_neurons} neurons x {recording.n_timepoints} samples")
print(f"windows: {windows.n_windows}; shape={windows.segments.shape}")
print(
    "reference timing: "
    f"history={WINDOW_CFG.effective_history_length / recording.fps:.3f}s, "
    f"target={WINDOW_CFG.target_length / recording.fps:.3f}s, "
    f"stride={WINDOW_CFG.stride / recording.fps:.3f}s, "
    f"adjacent-context overlap={1.0 - WINDOW_CFG.stride / WINDOW_CFG.effective_history_length:.1%}"
)
print(f"recording artifact: {stage_artifact_path(run, 'recording')}")
print(f"windows artifact: {stage_artifact_path(run, 'windows')}")
print_stage_status(run)
